Neural network:
Write a neural network model for a classification or regression task. Vary the number of hidden layers, number of neurons and other hyperparameters to find the best model for the problem.

In [17]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings

# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "mushrooms.csv"

# Load the latest version
df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS,"uciml/mushroom-classification",file_path)

print("First 5 records:")
display(df.head())
print(df.shape)

Using Colab cache for faster access to the 'mushroom-classification' dataset.
First 5 records:


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


(8124, 23)


In [18]:
# 1. Encode target column ('p' -> 1, 'e' -> 0)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['class'])

# 2. One-hot encode categorical features (drop first dummy to prevent multicollinearity)
X_encoded = pd.get_dummies(df.drop(columns=['class']), drop_first=True)

# 3. Convert features to float32 NumPy array for Keras compatibility
X = X_encoded.astype('float32').values

# 4. Split dataset (80% train, 20% test)
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("x_train shape (samples, features):", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

x_train shape (samples, features): (6499, 95)
y_train shape: (6499,)
x_test shape: (1625, 95)
y_test shape: (1625,)


In [19]:
def create_model(L, input_dim):
    """
    Creates a Sequential Keras model with 'L' hidden layers.
    Dynamically scales neuron counts between input_dim and 1.
    """
    model = Sequential()

    for j in range(1, L + 1):
        # Linearly decrease nodes across hidden layers
        nodes = int(input_dim - (input_dim - 1) * (j / (L + 1)))
        if nodes < 1:
            nodes = 1

        # First layer requires input_shape specification
        if j == 1:
            model.add(Dense(nodes, input_shape=(input_dim,), activation='relu'))
        else:
            model.add(Dense(nodes, activation='relu'))

        # Regularization layer
        model.add(Dropout(0.2))

    # Binary output layer (Sigmoid activation)
    model.add(Dense(1, activation='sigmoid'))

    # Compile model using Binary Crossentropy
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [20]:
results = []
accuracy = {}

# Preview how nodes scale dynamically across hidden layers for each configuration (L)
hidden_layers = [2, 4, 6, 8, 10]
input_dim = x_train.shape[1]  # 95 features from the mushroom dataset[cite: 1]

print("--- Dynamic Neuron Allocation per Hidden Layer ---")
for L in hidden_layers:
    print(f"Hidden Layers (L) = {L}")
    nodes_list = []
    for j in range(1, L + 1):
        # Linearly scale down nodes from input dimension to 1
        nodes = int(input_dim - (input_dim - 1) * (j / (L + 1)))
        if nodes < 1:
            nodes = 1
        nodes_list.append(nodes)
    print(f"Calculated Nodes: {nodes_list}\n")

for L in hidden_layers:
    print(f"\n==========================================")
    print(f"  Training model with {L} hidden layers")
    print(f"==========================================")

    # Create network architecture
    model = create_model(L, input_dim)

    # Train model
    history = model.fit(x_train,y_train,epochs=5,batch_size=128,validation_split=0.1,verbose=1)

    # Evaluate model on test split
    test_loss, test_accuracy = model.evaluate(x_test,y_test,verbose=0)

    # Store results
    results.append({
        'Hidden Layers': L,
        'Test Loss': test_loss,
        'Test Accuracy': test_accuracy,
        'Model': model,
        'History': history
    })

    print(f"\n---> Test Accuracy for {L} layers: {test_accuracy:.4f}")
    accuracy[L] = test_accuracy

--- Dynamic Neuron Allocation per Hidden Layer ---
Hidden Layers (L) = 2
Calculated Nodes: [63, 32]

Hidden Layers (L) = 4
Calculated Nodes: [76, 57, 38, 19]

Hidden Layers (L) = 6
Calculated Nodes: [81, 68, 54, 41, 27, 14]

Hidden Layers (L) = 8
Calculated Nodes: [84, 74, 63, 53, 42, 32, 21, 11]

Hidden Layers (L) = 10
Calculated Nodes: [86, 77, 69, 60, 52, 43, 35, 26, 18, 9]


  Training model with 2 hidden layers
Epoch 1/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8730 - loss: 0.3958 - val_accuracy: 0.9477 - val_loss: 0.1606
Epoch 2/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9708 - loss: 0.1032 - val_accuracy: 0.9831 - val_loss: 0.0478
Epoch 3/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9903 - loss: 0.0380 - val_accuracy: 0.9938 - val_loss: 0.0202
Epoch 4/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9952 - loss: 0.0185 - val_accuracy: 0.9954 - val_loss: 0.0121
Epoch 5/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9978 - loss: 0.0

In [21]:
print("\n" + "="*45)
print("        MODEL EVALUATION SUMMARY")
print("="*45)

for layers, acc in accuracy.items():
    print(f"Hidden Layers: {layers:2d}  |  Test Accuracy: {acc:.4f}")

# Find best model configuration
best_layers = max(accuracy, key=accuracy.get)
best_acc = accuracy[best_layers]

print("\n" + "-"*45)
print(f"Best Accuracy: {best_acc:.4f} achieved with {best_layers} hidden layers.")
print("-"*45)


        MODEL EVALUATION SUMMARY
Hidden Layers:  2  |  Test Accuracy: 0.9994
Hidden Layers:  4  |  Test Accuracy: 1.0000
Hidden Layers:  6  |  Test Accuracy: 1.0000
Hidden Layers:  8  |  Test Accuracy: 1.0000
Hidden Layers: 10  |  Test Accuracy: 1.0000

---------------------------------------------
Best Accuracy: 1.0000 achieved with 4 hidden layers.
---------------------------------------------
